[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/51_gdpo_loss_solution.ipynb)

# Solution: GDPO Loss

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
from torch import Tensor

In [ ]:
# ✅ SOLUTION

def gdpo_loss(logps: Tensor, old_logps: Tensor, ref_logps: Tensor,
              rewards: Tensor, group_ids: Tensor, completion_mask: Tensor,
              reward_weights: Tensor | None = None,
              clip_ratio: float = 0.2, beta: float = 0.1, eps: float = 1e-5) -> Tensor:
    if rewards.dim() == 1:
        rewards = rewards.unsqueeze(-1)

    num_rewards = rewards.shape[-1]
    if reward_weights is None:
        reward_weights = torch.ones(num_rewards, dtype=rewards.dtype, device=rewards.device) / num_rewards
    else:
        reward_weights = torch.as_tensor(reward_weights, dtype=rewards.dtype, device=rewards.device)

    per_reward_adv = torch.empty_like(rewards)
    for ridx in range(num_rewards):
        for gid in group_ids.unique():
            mask = group_ids == gid
            r_g = rewards[mask, ridx]
            per_reward_adv[mask, ridx] = (r_g - r_g.mean()) / (r_g.std(unbiased=False) + eps)

    agg_adv = (per_reward_adv * reward_weights.view(1, -1)).sum(dim=-1)
    agg_adv = (agg_adv - agg_adv.mean()) / (agg_adv.std(unbiased=False) + eps)
    agg_adv = agg_adv.detach().unsqueeze(1)

    delta_old = (logps - old_logps.detach()) * completion_mask
    ratio = torch.exp(delta_old)
    unclipped = ratio * agg_adv
    clipped = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * agg_adv
    delta_ref = (ref_logps.detach() - logps) * completion_mask
    kl = torch.exp(delta_ref) - delta_ref - 1.0

    token_objective = torch.minimum(unclipped, clipped) - beta * kl
    valid = completion_mask.sum(dim=-1).clamp_min(1.0)
    seq_objective = (token_objective * completion_mask).sum(dim=-1) / valid
    return -seq_objective.mean()


In [ ]:
# Demo
logps = torch.tensor([[-0.2, -0.1, -0.3], [-0.6, -0.5, -0.4], [-0.3, -0.8, -1.0], [-0.9, -1.1, -1.2]])
old_logps = torch.tensor([[-0.3, -0.2, -0.4], [-0.4, -0.4, -0.4], [-0.5, -0.6, -0.9], [-0.7, -1.0, -1.3]])
ref_logps = torch.tensor([[-0.25, -0.15, -0.35], [-0.55, -0.45, -0.45], [-0.35, -0.7, -0.95], [-0.8, -1.0, -1.1]])
rewards = torch.tensor([[1.0, 0.3], [0.8, 0.1], [0.3, 0.9], [0.1, 0.2]])
group_ids = torch.tensor([0, 0, 1, 1])
completion_mask = torch.tensor([[1, 1, 1], [1, 1, 0], [1, 1, 1], [1, 0, 0]], dtype=torch.float32)
reward_weights = torch.tensor([0.7, 0.3])
print('Loss:', gdpo_loss(logps, old_logps, ref_logps, rewards, group_ids, completion_mask, reward_weights=reward_weights))

In [ ]:
from torch_judge import check
check('gdpo_loss')